In [ ]:
import sys
sys.path.append("../")

In [ ]:
from src.dfg_ms_plexus.training_setup import *

In [ ]:
# root = Path('F:/DATA/dfg_plexus/')
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')
# root = Path("/mnt/Intenso/DATA/dfg_plexus/")

# df = pd.read_csv(root / "radiomics_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features___combined_SA___normalized.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features_shape_only___combined_SA___normalized_ICV.csv", delimiter="\t")
df = pd.read_csv(root / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

ft = df.drop(columns=['label', 'patID', 'DWI (0no,1yes)', 'LesionVolume', 'DiseaseDuration', 'EDSS'])
# ft = ft.drop(columns=['original_firstorder_Minimum', 'original_firstorder_Maximum', 'original_firstorder_Range'])
label = df['label'].astype(int)
label, class_mapping = get_labels_hc_cis_ms(label)  # get_labels_hc_ms, get_labels_full
print(f"{df.shape=}")
print(f"{ft.shape=}")
print(f"{label.shape=}")

In [ ]:
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(label),
                                     y=label)

# Dictionary mapping class labels to their corresponding weights
class_weight_dict = dict(zip(np.unique(label), class_weights))
class_weight_dict

In [ ]:
seeds = [0, 1, 2, 3, 4]

all_test_reports = []
all_train_reports = []
all_test_cms = []
all_train_cms = []
all_best_params = []

labels_sorted = np.sort(np.unique(label))
class_names = list(class_mapping.keys())

X_train, X_test, y_train, y_test = train_test_split(
    ft,
    label,
    test_size=0.3,
    random_state=42,
    stratify=label,
)

n_classes = np.unique(y_train).size
is_multiclass = n_classes > 2

for seed in seeds:
    print(f"\n========== Seed {seed} ==========")

    xgb = XGBClassifier(
        random_state=seed,
        objective='multi:softprob' if is_multiclass else 'binary:logistic',
        num_class=n_classes if is_multiclass else None,
        eval_metric='mlogloss' if is_multiclass else 'logloss',
        tree_method='hist',
        grow_policy='lossguide',
        device='cpu',
        n_jobs=12,
    )

    pipeline = Pipeline([
        ('var', VarianceThreshold(0.001)),
        ('scaler', StandardScaler()),
        # ('scaler', RobustScaler()),
        # ('feat', SelectKBest(score_func=f_classif)),
        ('classifier', xgb)
    ])

    # Best parameters found:  {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 2, 'classifier__max_leaves': 8, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 200, 'classifier__subsample': 0.9}

    param_grid = {
        # 'feat__k': [15, 20, 25],

        # --- Capacity
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [1, 2],
        "classifier__max_leaves": [8, 16],

        # --- Regularisation
        "classifier__min_child_weight": [1, 3],
        "classifier__gamma": [2.0, 5.0],
        "classifier__reg_lambda": [10.0, 50.0],

        # --- Randomness
        "classifier__subsample": [0.5, 0.7, 0.9],
        "classifier__colsample_bytree": [0.5, 0.7],

        # --- Optimisation
        "classifier__learning_rate": [0.01, 0.05],  # [0.05, 0.03, 0.01]

        # --- Imbalance
        # "classifier__scale_pos_weight": [scale_pos_weight],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scorer = make_scorer(f1_score, average='macro')

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring=scorer,
        n_jobs=12,
        verbose=1,
        refit=True  # Final fit on the whole training split with the best configuration
    )

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    grid_search.fit(X_train, y_train, classifier__sample_weight=sample_weights)

    import os
    import skops.io as sio

    os.makedirs('results', exist_ok=True)
    obj = sio.dump(grid_search, f'results/xgboost_clf___seed_{seed}___gridsearch.skops')

    all_best_params.append(grid_search.best_params_)

    y_train_pred = grid_search.predict(X_train)
    y_test_pred = grid_search.predict(X_test)

    train_report = classification_report(
        y_train,
        y_train_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    test_report = classification_report(
        y_test,
        y_test_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    all_train_reports.append(report_to_df(train_report))
    all_test_reports.append(report_to_df(test_report))

    all_train_cms.append(
        confusion_matrix(y_train, y_train_pred, labels=labels_sorted)
    )
    all_test_cms.append(
        confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
    )

In [ ]:
train_summary = aggregate_reports(all_train_reports)
test_summary = aggregate_reports(all_test_reports)

print("\n--- Train Set: Mean ± Std over seeds ---")
display(train_summary)

print("\n--- Test Set: Mean ± Std over seeds ---")
display(test_summary)

In [ ]:
train_cms = np.asarray(all_train_cms)
test_cms = np.asarray(all_test_cms)

train_cms_norm = np.asarray([normalize_cm(cm) for cm in train_cms])
test_cms_norm = np.asarray([normalize_cm(cm) for cm in test_cms])

train_cm_mean = train_cms_norm.mean(axis=0)
train_cm_std = train_cms_norm.std(axis=0)

test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    train_cm_mean,
    train_cm_std,
    "Trainset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import shap

# 1. Extract the best pipeline from your XGBoost GridSearch
best_pipeline = grid_search.best_estimator_

# 2. Isolate the preprocessing steps and the trained XGBoost model
var_thresh = best_pipeline.named_steps['var']
scaler = best_pipeline.named_steps['scaler']
xgb_model = best_pipeline.named_steps['classifier']

# 3. Push the training data through the preprocessing steps
# SHAP needs the data exactly as the model saw it (scaled and variance-filtered)
X_train_var = var_thresh.transform(X_train)
X_train_scaled = scaler.transform(X_train_var)

# 4. Recover the feature names that survived the VarianceThreshold
# Assuming 'ft' is your original pandas DataFrame
surviving_features = ft.columns[var_thresh.get_support()]

# Wrap the transformed data back into a DataFrame so the SHAP plot has the actual clinical names
X_train_shap = pd.DataFrame(X_train_scaled, columns=surviving_features)

# 5. Initialize the TreeExplainer
explainer = shap.TreeExplainer(xgb_model)

# 6. Calculate the SHAP values
# For multiclass XGBoost, this returns a list of array
shap_values = explainer.shap_values(X_train_shap)

# --- Step 7: Plot the Interactive Publication-Ready SHAP Plot ---

# 7a. Calculate the mean absolute SHAP values for each class
# This safely handles both 3D arrays (new SHAP) and lists (old SHAP)
if isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    # Shape is (n_samples, n_features, n_classes)
    # Average across samples (axis 0), resulting in (n_features, n_classes)
    mean_shap_matrix = np.abs(shap_values).mean(axis=0)
    # Split into a list of 1D arrays for each class
    mean_shap_values = [mean_shap_matrix[:, i] for i in range(mean_shap_matrix.shape[1])]
elif isinstance(shap_values, list):
    # Shape is a list of (n_samples, n_features) arrays
    mean_shap_values = [np.abs(sv).mean(axis=0) for sv in shap_values]
else:
    raise ValueError("Unexpected SHAP output format.")

# 7b. Create a DataFrame for easy sorting
df_shap = pd.DataFrame({
    'Feature': surviving_features,
    'Healthy Control (0)': mean_shap_values[0],
    'CIS (1)': mean_shap_values[1],
    'MS (2)': mean_shap_values[2]
})

# Calculate total importance to sort the top features
df_shap['Total_Importance'] = df_shap['Healthy Control (0)'] + df_shap['CIS (1)'] + df_shap['MS (2)']

# Keep only the top 15 features and sort ascending (so the largest is at the top of the horizontal bar chart)
df_shap = df_shap.sort_values(by='Total_Importance', ascending=True).tail(15)

# 7c. Build the Interactive Plotly Figure
fig = go.Figure()

# Define clinical colors for your classes
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
classes = ['Healthy Control (0)', 'CIS (1)', 'MS (2)']

# Add a stacked bar trace for each class
for i, cls in enumerate(classes):
    fig.add_trace(go.Bar(
        y=df_shap['Feature'],
        x=df_shap[cls],
        name=cls,
        orientation='h',
        marker=dict(color=colors[i])
    ))

# 7d. Formatting and Layout Tweaks
fig.update_layout(
    title=dict(
        text="Top Biomarkers for Predicting MS Disease Spectrum (SHAP)",
        font=dict(size=20),
        y=0.95
    ),
    xaxis_title="mean(|SHAP value|) (average impact on model output magnitude)",
    yaxis_title="Radiomic Feature",
    barmode='stack',       # Stacks the bars exactly like the SHAP package
    height=800,            # Plenty of room so feature names don't overlap
    width=1100,
    hovermode="y unified", # Shows the breakdown of all 3 classes simultaneously on hover
    legend=dict(
        title="Disease Spectrum Class",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    plot_bgcolor='rgba(245, 245, 245, 1)' # Light gray background for readability
)

# Add a subtle grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='white')

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import shap

# 1. Extract the best pipeline from your XGBoost GridSearch
best_pipeline = grid_search.best_estimator_

# 2. Isolate the preprocessing steps and the trained XGBoost model
var_thresh = best_pipeline.named_steps['var']
scaler = best_pipeline.named_steps['scaler']
xgb_model = best_pipeline.named_steps['classifier']

In [ ]:
X_train

In [ ]:
X_train.columns.to_list()

In [ ]:
var_thresh.variances_

In [ ]:
var_thresh.get_support()

In [ ]:
var_thresh.get_support().sum()

In [ ]:
for ft_name, var, support in zip(X_train.columns.to_list(), var_thresh.variances_, var_thresh.get_support()):
    print(f"{ft_name=}\t {var=}\t {support=}")

In [ ]:
print(f"{X_train.shape=}")

# 3. Push the training data through the preprocessing steps
# SHAP needs the data exactly as the model saw it (scaled and variance-filtered)
X_train_var = var_thresh.transform(X_train)
X_train_scaled = scaler.transform(X_train_var)

print(f"{X_train_scaled.shape=}")

# 4. Recover the feature names that survived the VarianceThreshold
# Assuming 'ft' is your original pandas DataFrame
surviving_features = ft.columns[var_thresh.get_support()]

# Wrap the transformed data back into a DataFrame so the SHAP plot has the actual clinical names
X_train_shap = pd.DataFrame(X_train_scaled, columns=surviving_features)

print(f"{X_train_shap.shape=}")

In [ ]:
[f for f in surviving_features if "shape" in f.lower()]

In [ ]:

# 5. Initialize the TreeExplainer
explainer = shap.TreeExplainer(xgb_model)

# 6. Calculate the SHAP values
# For multiclass XGBoost, this returns a list of array
shap_values = explainer.shap_values(X_train_shap)

# --- Step 7: Plot the Interactive Publication-Ready SHAP Plot ---

# 7a. Calculate the mean absolute SHAP values for each class
# This safely handles both 3D arrays (new SHAP) and lists (old SHAP)
if isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    # Shape is (n_samples, n_features, n_classes)
    # Average across samples (axis 0), resulting in (n_features, n_classes)
    mean_shap_matrix = np.abs(shap_values).mean(axis=0)
    # Split into a list of 1D arrays for each class
    mean_shap_values = [mean_shap_matrix[:, i] for i in range(mean_shap_matrix.shape[1])]
elif isinstance(shap_values, list):
    # Shape is a list of (n_samples, n_features) arrays
    mean_shap_values = [np.abs(sv).mean(axis=0) for sv in shap_values]
else:
    raise ValueError("Unexpected SHAP output format.")

# 7b. Create a DataFrame for easy sorting
df_shap = pd.DataFrame({
    "Feature": surviving_features,
    "Healthy Control (0)": mean_shap_values[0],
    "CIS (1)": mean_shap_values[1],
    "MS (2)": mean_shap_values[2],
})

# ------------------------------------------------------------
# Filter to a specific feature group
# Example: Pyradiomics shape features
# ------------------------------------------------------------

shape_mask = df_shap["Feature"].str.contains(
    "shape",
    case=False,
    regex=False
)

df_shap = df_shap[shape_mask].copy()

if df_shap.empty:
    raise ValueError(
        "No shape features found. Check the exact feature names in surviving_features."
    )

# Calculate total importance to sort the top features
df_shap["Total_Importance"] = (
    df_shap["Healthy Control (0)"]
    + df_shap["CIS (1)"]
    + df_shap["MS (2)"]
)

# Keep only the top shape features
df_shap = (
    df_shap
    .sort_values(by="Total_Importance", ascending=True)
    .tail(15)
)

# 7c. Build the Interactive Plotly Figure
fig = go.Figure()

# Define clinical colors for your classes
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
classes = ['Healthy Control (0)', 'CIS (1)', 'MS (2)']

# Add a stacked bar trace for each class
for i, cls in enumerate(classes):
    fig.add_trace(go.Bar(
        y=df_shap['Feature'],
        x=df_shap[cls],
        name=cls,
        orientation='h',
        marker=dict(color=colors[i])
    ))

# 7d. Formatting and Layout Tweaks
fig.update_layout(
    title=dict(
        text="Top Biomarkers for Predicting MS Disease Spectrum (SHAP)",
        font=dict(size=20),
        y=0.95
    ),
    xaxis_title="mean(|SHAP value|) (average impact on model output magnitude)",
    yaxis_title="Radiomic Feature",
    barmode='stack',       # Stacks the bars exactly like the SHAP package
    height=800,            # Plenty of room so feature names don't overlap
    width=1100,
    hovermode="y unified", # Shows the breakdown of all 3 classes simultaneously on hover
    legend=dict(
        title="Disease Spectrum Class",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    plot_bgcolor='rgba(245, 245, 245, 1)' # Light gray background for readability
)

# Add a subtle grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='white')

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import shap
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Helper: make SHAP output class-specific
# ------------------------------------------------------------

class_names = ["Healthy Control (0)", "CIS (1)", "MS (2)"]

def get_class_shap_values(shap_values, class_idx, n_features):
    """
    Returns SHAP values with shape (n_samples, n_features)
    for one class, handling both old and new SHAP formats.
    """
    if isinstance(shap_values, list):
        return shap_values[class_idx]

    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        # Common new SHAP format: (n_samples, n_features, n_classes)
        if shap_values.shape[1] == n_features:
            return shap_values[:, :, class_idx]

        # Alternative format: (n_classes, n_samples, n_features)
        if shap_values.shape[2] == n_features:
            return shap_values[class_idx, :, :]

    raise ValueError(f"Unexpected SHAP shape: {np.shape(shap_values)}")


# ------------------------------------------------------------
# Use unscaled feature values for display/coloring
# SHAP values were computed on scaled data, but original values
# are easier to interpret in beeswarm/dependence plots.
# ------------------------------------------------------------

if isinstance(X_train, pd.DataFrame):
    X_train_raw_df = X_train.copy()
else:
    X_train_raw_df = pd.DataFrame(X_train, columns=ft.columns)

X_train_display = X_train_raw_df.loc[:, surviving_features].reset_index(drop=True)
X_train_shap = X_train_shap.reset_index(drop=True)

n_features = X_train_shap.shape[1]

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 0  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
# plt.savefig("lgbm_clf___shap_beeswarm___HC.svg", bbox_inches="tight", format="svg")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 1  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
# plt.savefig("lgbm_clf___shap_beeswarm___CIS.svg", bbox_inches="tight", format="svg")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 2  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
# plt.savefig("lgbm_clf___shap_beeswarm___MS.svg", bbox_inches="tight", format="svg")
plt.show()